# OpenVINO Tokenizers: Incorporate Text Processing Into OpenVINO Pipelines

<center><img src="https://github.com/openvinotoolkit/openvino_notebooks/assets/51917466/047f9167-a4ef-4d3d-a33b-d124541f9e2c"></center>

OpenVINO Tokenizers is an OpenVINO extension and a Python library designed to streamline tokenizer conversion for seamless integration into your projects. It supports Python and C++ environments and is compatible with all major platforms: Linux, Windows, and MacOS.


#### Table of contents:
- [Tokenization Basics](#Tokenization-Basics)
- [Acquiring OpenVINO Tokenizers](#Acquiring-OpenVINO-Tokenizers)
    - [Convert Tokenizer from HuggingFace Hub with CLI Tool](#Convert-Tokenizer-from_HuggingFace-Hub-with-CLI-Tool)
    - [Convert Tokenizer from HuggingFace Hub with Python API](#Convert-Tokenizer-from-HuggingFace-Hub-with-Python-API)
- [Text Generation Pipeline with OpenVINO Tokenizers](#Text-Generation-Pipeline-with-OpenVINO-Tokenizers)
- [Text Generation Pipeline with OpenVINO GenAI and OpenVINO Tokenizers](#text-generation-pipeline-with-openvino-genai-and-openvino-tokenizers)
- [Merge Tokenizer into a Model](#Merge-Tokenizer-into-a-Model)
- [Conclusion](#Conclusion)
- [Links](#Links)


## Tokenization Basics

[back to top ⬆️](#Table-of-contents:)

One does not simply put text into a neural network, only numbers. The process of transforming text into a sequence of numbers is called **tokenization**. It usually contains several steps that transform the original string, splitting it into parts - tokens - with an associated number in a dictionary. You can check the [interactive GPT-4 tokenizer](https://platform.openai.com/tokenizer) to gain an intuitive understanding of the principles of tokenizer work.

<center><img src="https://github.com/openvinotoolkit/openvino_notebooks/assets/51917466/55cc2420-da2a-4022-bae3-84d566a68e4c"></center>

There are two important points in the tokenizer-model relation:
1. Every neural network with text input is paired with a tokenizer and _cannot be used without it_.
2. To reproduce the model's accuracy on a specific task, it is essential to _utilize the same tokenizer employed during the model training_.

That is why almost all model repositories on [HuggingFace Hub](https://HuggingFace.co/models) also contain tokenizer files (`tokenizer.json`, `vocab.txt`, `merges.txt`, etc.).

The process of transforming a sequence of numbers into a string is called **detokenization**. Detokenizer can share the token dictionary with a tokenizer, like any LLM chat model, or operate with an entirely distinct dictionary. For instance, translation models dealing with different source and target languages often necessitate separate dictionaries.

<center><img src="https://github.com/openvinotoolkit/openvino_notebooks/assets/51917466/942f1692-b7f2-4aca-93ef-32e9866c204f"></center>

Some tasks only need a tokenizer, like text classification, named entity recognition, question answering, and feature extraction. On the other hand, for tasks such as text generation, chat, translation, and abstractive summarization, both a tokenizer and a detokenizer are required.
### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/openvino-tokenizers/openvino-tokenizers.ipynb" />


## Acquiring OpenVINO Tokenizers
[back to top ⬆️](#Table-of-contents:)

OpenVINO Tokenizers Python library allows you to convert HuggingFace tokenizers into OpenVINO models. The following cell installs OpenVINO nightly builds with the `connect_models` sink fix and the other required dependencies.

In [1]:
%pip install -q -U --pre "openvino>=2024.3.0" openvino-tokenizers[transformers] openvino-genai --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install "torch==2.8" "torchvision==0.23.0" "transformers>=4.51.0,<5.0" accelerate --extra-index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple, https://download.pytorch.org/whl/cpu


  Using cached torch-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)


  Using cached torchvision-0.23.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)


  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformers]


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("openvino-tokenizers.ipynb")

In [3]:
from pathlib import Path

tokenizer_dir = Path("tokenizer/")
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

### Convert Tokenizer from HuggingFace Hub with CLI Tool
[back to top ⬆️](#Table-of-contents:)

The first way is to use the CLI utility, bundled with OpenVINO Tokenizers. Use `--with-detokenizer` flag to add the detokenizer model to the output. By setting `--clean-up-tokenization-spaces=False` we ensure that the detokenizer correctly decodes a code-generation model output. `--trust-remote-code` flag works the same way as passing `trust_remote_code=True` to `AutoTokenizer.from_pretrained` constructor.

In [4]:
if not tokenizer_dir.exists():
    !convert_tokenizer $model_id --with-detokenizer -o $tokenizer_dir

> ⚠️ If you have any problems with the command above on MacOS, try to [install tbb](https://formulae.brew.sh/formula/tbb#default).

The result is two OpenVINO models: `openvino_tokenizer` and `openvino_detokenizer`. Both can be interacted with using `read_model`, `compile_model` and `save_model`, similar to any other OpenVINO model.

### Convert Tokenizer from HuggingFace Hub with Python API
[back to top ⬆️](#Table-of-contents:)

The other method is to pass HuggingFace `hf_tokenizer` object to `convert_tokenizer` function:

In [5]:
from transformers import AutoTokenizer
from openvino_tokenizers import convert_tokenizer

hf_tokenizer = AutoTokenizer.from_pretrained(model_id)
ov_tokenizer, ov_detokenizer = convert_tokenizer(hf_tokenizer, with_detokenizer=True)
ov_tokenizer, ov_detokenizer

/workspace/.venvs/cpu/ec74fa9a5736664e/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

(<Model: 'tokenizer'
 inputs[
 <ConstOutput: names[Parameter_1] shape[?] type: string>
 ]
 outputs[
 <ConstOutput: names[input_ids] shape[?,?] type: i64>,
 <ConstOutput: names[attention_mask] shape[?,?] type: i64>
 ]>,
 <Model: 'detokenizer'
 inputs[
 <ConstOutput: names[Parameter_143] shape[?,?] type: i64>
 ]
 outputs[
 <ConstOutput: names[string_output, Result_165] shape[?] type: string>
 ]>)

That way you get OpenVINO model objects. Use `save_model` function from OpenVINO to reuse converted tokenizers later:

> ⚠️ Import `openvino_tokenizers` will add all tokenizer-related operations to OpenVINO, after which you can work with saved tokenizers and detokenizers.

In [6]:
import openvino as ov

# This import is needed to add all tokenizer-related operations to OpenVINO
import openvino_tokenizers  # noqa: F401

ov.save_model(ov_tokenizer, tokenizer_dir / "openvino_tokenizer.xml")
ov.save_model(ov_detokenizer, tokenizer_dir / "openvino_detokenizer.xml")

To use the tokenizer, compile the converted model and input a list of strings.  It's essential to be aware that not all original tokenizers support multiple strings (also called batches) as input.  This limitation arises from the requirement for all resulting number sequences to maintain the same length. To address this, a padding token must be specified, which will be appended to shorter tokenized strings. In cases where no padding token is determined in the original tokenizer, OpenVINO Tokenizers defaults to using $0$ for padding.  Presently, _only right-side padding is supported_, typically used for classification tasks, but not suitable for text generation.

In [7]:
tokenizer, detokenizer = ov.compile_model(ov_tokenizer), ov.compile_model(ov_detokenizer)
test_strings = ["Test", "strings"]

token_ids = tokenizer(test_strings)["input_ids"]
print(f"Token ids: {token_ids}")

detokenized_text = detokenizer(token_ids)["string_output"]
print(f"Detokenized text: {detokenized_text}")

[16:30:16.6666]D[plugin.cpp:289][AUTO] deviceNameWithID:CPU, defaultDeviceID:, uniqueName:CPU_
[16:30:16.6682]I[plugin.cpp:479][AUTO] device:CPU, config:LOG_LEVEL=LOG_NONE
[16:30:16.6682]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT=LATENCY
[16:30:16.6682]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT_NUM_REQUESTS=0
[16:30:16.6682]I[plugin.cpp:479][AUTO] device:CPU, config:PERF_COUNT=NO
[16:30:16.6682]I[plugin.cpp:484][AUTO] device:CPU, priority:0
[16:30:16.6682]I[plugin.cpp:510][AUTO] Original device list() size to load the model: 0
[16:30:16.6682]I[schedule.cpp:17][AUTO] scheduler starting
[16:30:16.6682]I[auto_schedule.cpp:163][AUTO] select device:CPU
[16:30:16.6687]I[auto_schedule.cpp:284][AUTO] Only one device or multiple devices of the same type will use passthrough compiled model
[16:30:16.7155]I[auto_schedule.cpp:341][AUTO] Device: [CPU]: Compile model took 46.818408 ms
[16:30:16.7162]I[auto_schedule.cpp:118][AUTO] device:CPU compiling model finished


We can compare the result of converted (de)tokenizer with the original one:

In [8]:
hf_token_ids = hf_tokenizer(test_strings).input_ids
print(f"Token ids: {hf_token_ids}")

hf_detokenized_text = hf_tokenizer.batch_decode(hf_token_ids)
print(f"Detokenized text: {hf_detokenized_text}")

Token ids: [[1, 4321], [1, 6031]]
Detokenized text: ['<s> Test', '<s> strings']


## Text Generation Pipeline with OpenVINO Tokenizers
[back to top ⬆️](#Table-of-contents:)

Let's build a text generation pipeline with OpenVINO Tokenizers and minimal dependencies. To obtain an OpenVINO model we will use the Optimum library. The latest version allows you to get a so-called [stateful model](https://docs.openvino.ai/2024/openvino-workflow/running-inference/stateful-models.html).

The original `TinyLlama-1.1B-intermediate-step-1431k-3T` model is 4.4Gb. To reduce network and disk usage we will load a converted model which has also been compressed to `int8`. The original conversion command is commented.

In [9]:
model_dir = Path(Path(model_id).name)

if not model_dir.exists():
    # Converting the original model
    # %pip install -U "https://codeload.github.com/huggingface/optimum-intel/tar.gz/HEAD" "nncf>=2.8.0" onnx
    # %optimum-cli export openvino -m $model_id --task text-generation-with-past $model_dir

    # Load already converted model
    from huggingface_hub import hf_hub_download

    hf_hub_download(
        "chgk13/TinyLlama-1.1B-intermediate-step-1431k-3T",
        filename="openvino_model.xml",
        local_dir=model_dir,
    )
    hf_hub_download(
        "chgk13/TinyLlama-1.1B-intermediate-step-1431k-3T",
        filename="openvino_model.bin",
        local_dir=model_dir,
    )

In [10]:
import numpy as np
from tqdm.notebook import trange
from pathlib import Path
from openvino_tokenizers.constants import EOS_TOKEN_ID_NAME

core = ov.Core()

ov_model = core.read_model(model_dir / "openvino_model.xml")
compiled_model = core.compile_model(ov_model)
infer_request = compiled_model.create_infer_request()

[16:30:16.8881]D[plugin.cpp:289][AUTO] deviceNameWithID:CPU, defaultDeviceID:, uniqueName:CPU_
[16:30:16.8882]I[plugin.cpp:479][AUTO] device:CPU, config:LOG_LEVEL=LOG_NONE
[16:30:16.8882]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT=LATENCY
[16:30:16.8882]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT_NUM_REQUESTS=0
[16:30:16.8882]I[plugin.cpp:479][AUTO] device:CPU, config:PERF_COUNT=NO
[16:30:16.8882]I[plugin.cpp:484][AUTO] device:CPU, priority:0
[16:30:16.8882]I[plugin.cpp:510][AUTO] Original device list() size to load the model: 0
[16:30:16.8882]I[schedule.cpp:17][AUTO] scheduler starting
[16:30:16.8882]I[auto_schedule.cpp:163][AUTO] select device:CPU
[16:30:16.9016]I[auto_schedule.cpp:284][AUTO] Only one device or multiple devices of the same type will use passthrough compiled model


[16:30:17.7440]I[auto_schedule.cpp:341][AUTO] Device: [CPU]: Compile model took 842.297434 ms
[16:30:17.7445]I[auto_schedule.cpp:118][AUTO] device:CPU compiling model finished
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:SUPPORTED_PROPERTIES=SUPPORTED_PROPERTIES NETWORK_NAME OPTIMAL_NUMBER_OF_INFER_REQUESTS NUM_STREAMS INFERENCE_NUM_THREADS PERF_COUNT INFERENCE_PRECISION_HINT PERFORMANCE_HINT EXECUTION_MODE_HINT PERFORMANCE_HINT_NUM_REQUESTS ENABLE_
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:NETWORK_NAME=Model0
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:OPTIMAL_NUMBER_OF_INFER_REQUESTS=1
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:NUM_STREAMS=1
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:INFERENCE_NUM_THREADS=32
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:PERF_COUNT=NO
[16:30:17.7445]D[auto_schedule.cpp:124][AUTO] device:CPU, GetConfig:INFEREN

The `infer_request` object provides control over the model's state - a Key-Value cache that speeds up inference by reducing computations. Multiple inference requests can be created, and each request maintains a distinct and separate state.

In [11]:
text_input = ["Quick brown fox jumped"]

model_input = {name.any_name: output for name, output in tokenizer(text_input).items()}

if "position_ids" in (input.any_name for input in infer_request.model_inputs):
    model_input["position_ids"] = np.arange(model_input["input_ids"].shape[1], dtype=np.int64)[np.newaxis, :]

# No beam search, set idx to 0
model_input["beam_idx"] = np.array([0], dtype=np.int32)

# End of sentence token is that model signifies the end of text generation
# Read EOS token ID from rt_info of tokenizer/detokenizer ov.Model object
eos_token = ov_tokenizer.get_rt_info(EOS_TOKEN_ID_NAME).value

tokens_result = np.array([[]], dtype=np.int64)

# Reset KV cache inside the model before inference
infer_request.reset_state()
max_infer = 5

for _ in trange(max_infer):
    infer_request.start_async(model_input)
    infer_request.wait()

    output_tensor = infer_request.get_output_tensor()

    # Get the most probable token
    token_indices = np.argmax(output_tensor.data, axis=-1)
    output_token = token_indices[:, -1:]

    # Concatenate previous tokens result with newly generated token
    tokens_result = np.hstack((tokens_result, output_token))
    if output_token[0, 0] == eos_token:
        break

    # Prepare input for the next inference iteration
    model_input["input_ids"] = output_token
    model_input["attention_mask"] = np.hstack((model_input["attention_mask"].data, [[1]]))
    model_input["position_ids"] = np.hstack(
        (
            model_input["position_ids"].data,
            [[model_input["position_ids"].data.shape[-1]]],
        )
    )


text_result = detokenizer(tokens_result)["string_output"]
print(f"Prompt:\n{text_input[0]}")
print(f"Generated:\n{text_result[0]}")

  0%|          | 0/5 [00:00<?, ?it/s]

Prompt:
Quick brown fox jumped
Generated:
over the fence.


## Text Generation Pipeline with OpenVINO GenAI and OpenVINO Tokenizers
[back to top ⬆️](#Table-of-contents:)

[OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) is a flavor of OpenVINO, aiming to simplify running inference of generative AI models. It hides the complexity of the generation process and minimizes the amount of code required.
OpenVINO GenAI depends on [OpenVINO](https://github.com/openvinotoolkit/openvino) and [OpenVINO Tokenizers](https://github.com/openvinotoolkit/openvino_tokenizers).

Firstly we need to create a pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for text generation using LLM in OpenVINO GenAI API. You can construct it straight away from the folder where both converted model and tokenizer are located, e.g. `ov_genai.LLMPipeline(model_and_tokenizer_path)`. 

As the model and tokenizer are located in different directories, we create a `ov_genai.Tokenizer` object by providing the path to saved tokenizer. Then we will provide directory with model, tokenizer object and device for `LLMPipeline`. Lastly we run `generate` method and get the output in text format.

Additionally, we can configure parameters for decoding. We can get the default config with `get_generation_config()`, setup parameters, and apply the updated version with `set_generation_config(config)` or put config directly to `generate()`. It's also possible to specify the needed options just as inputs in the `generate()` method, as shown below, e.g. we can add `max_new_tokens` to stop generation if a specified number of tokens is generated and the end of generation is not reached.

Let's build the same text generation pipeline, but with simplified Python [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai/blob/master/src/README.md). We will use the same model and tokenizer downloaded in previous steps.

In [12]:
import openvino_genai as ov_genai

genai_tokenizer = ov_genai.Tokenizer(str(tokenizer_dir))
pipe = ov_genai.LLMPipeline(str(model_dir), genai_tokenizer, "CPU")

result = pipe.generate(text_input[0], max_new_tokens=max_infer)

print(f"Prompt:\n{text_input[0]}")
print(f"Generated:\n{result}")

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Prompt:
Quick brown fox jumped
Generated:
over the lazy dog.


## Merge Tokenizer into a Model
[back to top ⬆️](#Table-of-contents:)

Packages like `tensorflow-text` offer the convenience of integrating text processing directly into the model, streamlining both distribution and usage.  Similarly, with OpenVINO Tokenizers, you can create models that combine a converted tokenizer and a model. It's important to note that not all scenarios benefit from this merge. In cases where a tokenizer is used once and a model is inferred multiple times, as seen in the earlier text generation example, maintaining a separate (de)tokenizer and model is advisable to prevent unnecessary tokenization-detokenization cycles during inference. Conversely, if both a tokenizer and a model are used once in each pipeline inference, merging simplifies the workflow and aids in avoiding the creation of intermediate objects:

<center><img src="https://github.com/openvinotoolkit/openvino_notebooks/assets/51917466/d4ece285-e445-4b76-a1ab-356427900860"></center>

The OpenVINO Python API allows you to avoid this by using the `share_inputs` option during inference, but it requires additional input from a developer every time the model is inferred. Combining the models and tokenizers simplifies memory management.
Moreover, after the combining models inputs have changed - original model has three inputs (`input_ids`, `attention_mask`, `token_type_ids`) and combined model has only one input for text input prompt.

In [13]:
from cmd_helper import optimum_cli

model_id = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
model_dir = Path(Path(model_id).name)

if not model_dir.exists():
    %pip install -qU "https://codeload.github.com/huggingface/optimum-intel/tar.gz/HEAD" --extra-index-url https://download.pytorch.org/whl/cpu
    optimum_cli(model_id, model_dir, additional_args={"task": "text-classification"})

Note: you may need to restart the kernel to use updated packages.


**Export command:**

`optimum-cli export openvino --model mrm8488/bert-tiny-finetuned-sms-spam-detection bert-tiny-finetuned-sms-spam-detection --task text-classification`

In [14]:
from openvino_tokenizers import connect_models

core = ov.Core()
text_input = ["Free money!!!"]

ov_tokenizer = core.read_model(model_dir / "openvino_tokenizer.xml")
ov_model = core.read_model(model_dir / "openvino_model.xml")
combined_model = connect_models(ov_tokenizer, ov_model)
ov.save_model(combined_model, model_dir / "combined_openvino_model.xml")

print("Original OpenVINO model inputs:")
for input in ov_model.inputs:
    print(input)

print("\nCombined OpenVINO model inputs:")
for input in combined_model.inputs:
    print(input)

compiled_combined_model = core.compile_model(combined_model)
openvino_output = compiled_combined_model(text_input)

print(f"\nLogits: {openvino_output['logits']}")

Original OpenVINO model inputs:
<Output: names[input_ids] shape[?,?] type: i64>
<Output: names[attention_mask] shape[?,?] type: i64>
<Output: names[token_type_ids] shape[?,?] type: i64>

Combined OpenVINO model inputs:
<Output: names[Parameter_5512] shape[?] type: string>
[16:31:35.2918]D[plugin.cpp:289][AUTO] deviceNameWithID:CPU, defaultDeviceID:, uniqueName:CPU_
[16:31:35.2919]I[plugin.cpp:479][AUTO] device:CPU, config:LOG_LEVEL=LOG_NONE
[16:31:35.2919]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT=LATENCY
[16:31:35.2919]I[plugin.cpp:479][AUTO] device:CPU, config:PERFORMANCE_HINT_NUM_REQUESTS=0
[16:31:35.2919]I[plugin.cpp:479][AUTO] device:CPU, config:PERF_COUNT=NO
[16:31:35.2919]I[plugin.cpp:484][AUTO] device:CPU, priority:0
[16:31:35.2919]I[plugin.cpp:510][AUTO] Original device list() size to load the model: 0
[16:31:35.2919]I[schedule.cpp:17][AUTO] scheduler starting
[16:31:35.2920]I[auto_schedule.cpp:163][AUTO] select device:CPU
[16:31:35.2928]I[auto_schedule.cpp:28

RuntimeError: Exception from src/inference/src/cpp/infer_request.cpp:224:
Exception from src/plugins/intel_cpu/src/graph.cpp:1628:
Node ReadValue_5627 of type ReadValue
Check 'outputNode' failed at src/plugins/intel_cpu/src/nodes/memory.cpp:487:
ReadValue node with name 'ReadValue_5627'  doesn't have sibling output




## Conclusion
[back to top ⬆️](#Table-of-contents:)

The OpenVINO Tokenizers integrate text processing operations into the OpenVINO ecosystem. Enabling the conversion of HuggingFace tokenizers into OpenVINO models, the library allows efficient deployment of deep learning pipelines across varied environments. The feature of combining tokenizers and models not only simplifies memory management but also helps to streamline model usage and deployment.


## Links
[back to top ⬆️](#Table-of-contents:)

- [Installation instructions for different environments](https://github.com/openvinotoolkit/openvino_tokenizers?tab=readme-ov-file#installation)
- [Supported Tokenizer Types](https://github.com/openvinotoolkit/openvino_tokenizers?tab=readme-ov-file#supported-tokenizer-types)
- [OpenVINO.GenAI repository with the C++ example of OpenVINO Tokenizers usage](https://github.com/openvinotoolkit/openvino.genai/tree/master/samples/cpp/greedy_causal_lm)
- [HuggingFace Tokenizers Comparison Table](https://github.com/openvinotoolkit/openvino_tokenizers?tab=readme-ov-file#output-match-by-model)